In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install faiss-cpu
!pip install pypdf
!pip install openai
!pip install tiktoken

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import sys
print(sys.executable)

/usr/bin/python3


In [3]:
!which python3

/usr/bin/python3


In [4]:
!pip install --break-system-packages langchain langchain-community langchain-openai faiss-cpu pypdf openai tiktoken

Defaulting to user installation because normal site-packages is not writeable


In [5]:
import langchain
print(langchain.__version__)

1.3.2


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

/home/nineleaps/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = TextLoader("rag_notes.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = splitter.split_documents(docs)
print(f"Number of chunks: {len(chunks)}")
print(chunks[0].page_content)

Number of chunks: 1
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with large language models.

Instead of relying only on the model's internal knowledge, RAG retrieves relevant information from external documents before generating a response.

Benefits of RAG:
1. Reduces hallucinations.
2. Improves factual accuracy.
3. Allows access to updated information.
4. Enables question answering over private documents.


/tmp/ipykernel_67856/3466753485.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [8]:
!pip install --break-system-packages sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)
retriever = vectorstore.as_retriever()
print("FAISS Vector Store Created Successfully!")

/tmp/ipykernel_67856/2250291838.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 3149.06it/s]


FAISS Vector Store Created Successfully!


In [10]:
chat_history = []

In [11]:
def add_to_history(role, content):
    chat_history.append({
        "role": role,
        "content": content
    })

In [12]:
add_to_history("user", "What is RAG?")
add_to_history("assistant", "RAG stands for Retrieval-Augmented Generation.")
print(chat_history)

[{'role': 'user', 'content': 'What is RAG?'}, {'role': 'assistant', 'content': 'RAG stands for Retrieval-Augmented Generation.'}]


In [13]:
def show_history():
    for msg in chat_history:
        print(f"{msg['role'].upper()}: {msg['content']}")

In [14]:
show_history()

USER: What is RAG?
ASSISTANT: RAG stands for Retrieval-Augmented Generation.


In [15]:
def rewrite_question(question):

    if len(chat_history) == 0:
        return question

    last_question = None

    for msg in reversed(chat_history):
        if msg["role"] == "user":
            last_question = msg["content"]
            break

    if "it" in question.lower() and last_question:
        return f"Can you explain '{last_question}' in more detail?"

    return question

In [16]:
chat_history = []

add_to_history("user", "What is RAG?")

print(
    rewrite_question(
        "Can you explain it more?"
    )
)

Can you explain 'What is RAG?' in more detail?


In [17]:
from tools import get_current_time
print(get_current_time())

15:23:01


In [18]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Returns the current system time",

            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

In [19]:
from tools import get_current_time
def execute_tool(tool_name):
    if tool_name == "get_current_time":
        return get_current_time()
    return "Tool not found"

In [20]:
result = execute_tool(
    "get_current_time"
)
print(result)

15:23:01


In [21]:
def tool_router(user_query):
    user_query = user_query.lower()
    if "time" in user_query:
        return {
            "tool_call": True,
            "tool_name": "get_current_time"
        }
    return {
        "tool_call": False
    }

In [22]:
response = tool_router(
    "What is the current time?"
)
print(response)

{'tool_call': True, 'tool_name': 'get_current_time'}


In [23]:
user_query = "What is the current time?"
decision = tool_router(user_query)
if decision["tool_call"]:
    result = execute_tool(
        decision["tool_name"]
    )
    answer = (
        f"The current time is {result}"
    )
else:
    answer = "No tool needed"
print(answer)

The current time is 15:23:01


In [24]:
def retrieve_context(query):
    docs = retriever.invoke(query)
    if len(docs) == 0:
        return None
    return docs[0].page_content

In [25]:
print(
    retrieve_context(
        "What is RAG?"
    )
)

Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with large language models.

Instead of relying only on the model's internal knowledge, RAG retrieves relevant information from external documents before generating a response.

Benefits of RAG:
1. Reduces hallucinations.
2. Improves factual accuracy.
3. Allows access to updated information.
4. Enables question answering over private documents.


In [26]:
def retrieve_context(query):

    docs = retriever.invoke(query)

    if not docs:
        return None

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    return context

In [27]:
print(retrieve_context("What is RAG?"))

Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with large language models.

Instead of relying only on the model's internal knowledge, RAG retrieves relevant information from external documents before generating a response.

Benefits of RAG:
1. Reduces hallucinations.
2. Improves factual accuracy.
3. Allows access to updated information.
4. Enables question answering over private documents.


In [28]:
def route_query(user_query):

    if "time" in user_query.lower():
        return "TOOL"

    return "RAG"

In [29]:
print(route_query("What is RAG?"))
print(route_query("What is the current time?"))

RAG
TOOL


In [30]:
print(retrieve_context("What is RAG?"))

Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with large language models.

Instead of relying only on the model's internal knowledge, RAG retrieves relevant information from external documents before generating a response.

Benefits of RAG:
1. Reduces hallucinations.
2. Improves factual accuracy.
3. Allows access to updated information.
4. Enables question answering over private documents.


In [31]:
print(execute_tool("get_current_time"))

15:23:01


In [32]:
chat_history = []

while True:

    user_input = input("\nYou: ")

    if user_input.lower() == "exit":
        print("Goodbye!")
        break

    chat_history.append({
        "role": "user",
        "content": user_input
    })

    route = route_query(user_input)

    if route == "TOOL":

        result = execute_tool("get_current_time")

        answer = f"The current time is {result}"

    else:

        answer = retrieve_context(user_input)

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    print("\nAssistant:")
    print(answer)


You:  What is RAG?



Assistant:
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with large language models.

Instead of relying only on the model's internal knowledge, RAG retrieves relevant information from external documents before generating a response.

Benefits of RAG:
1. Reduces hallucinations.
2. Improves factual accuracy.
3. Allows access to updated information.
4. Enables question answering over private documents.



You:  What is the current time?



Assistant:
The current time is 15:23:31



You:  exit


Goodbye!
